# C3 Step 9 — E009: does MORE training data help? (the E008 recipe, re-run)

**The question (Sathmina, 30 Sep 2026):** did the new batch of data improve the output?
A zero-shot model cannot answer that. It never trains on the data, so more data can only change what is *measured*, not what the model *does*.
This notebook retrains and compares.

**The only thing that changes vs E008 is the training set:**

| arm | training clips | minutes | overlap |
|---|---|---|---|
| `small` = E008's exact set | 21 | 36.4 | 2.7 % |
| `big` = small + new batch R0029–R0052 | 45 | 115.2 | 9.1 % |

DEV = R0023–R0026 (10.7 min, early stopping only, same as E008).
TEST = the 2 IRD YouTube clips (5.1 min, 12.0 % overlap), frozen, identical to E005/E008, scored once per model.

**Seeds:** each arm is trained 3 times (seeds 0, 1, 2). The spread across seeds is the noise floor: a gap smaller than that spread means nothing.

**Pre-registered decision rule (written 30 Sep 2026, before running):**
- Primary number: TEST pooled DER, collar 0.0.
- "More data helped" only if **every** `big` run beats **every** `small` run (exact permutation p = 1/20 = 0.05 with 3 v 3). The reverse means "more data hurt". Anything else means "no detectable difference at this test size".
- Limits, stated now: TEST is 2 clips / 5.1 min. Seeds measure training noise, not test-sample noise. `big` adds data that is both MORE and DIFFERENT (much more overlap), so this design cannot separate amount from type.

**Pre-registered expectation:** E008 hurt TEST through missed speech (1.48 → 3.63). It was trained on near-zero-overlap audio and became conservative. The new batch's overlap matches TEST, so I expect `big` to undo most of that missed-speech damage and beat `small`. I do **not** expect TEST confusion (~11 %) to move, because three experiments agree segmentation can't fix it. So `big` may only climb back to the pretrained level (14.35), not below it. Nothing in Part A is expected to approach community-1 zero-shot (7.50).

**Part B (optional, runs after Part A is saved):** the same small-vs-big test inside **community-1**, the pipeline the component actually uses. It fine-tunes community-1's own segmentation model and swaps it in. Two controls run first: community-1 must reproduce E004's TEST 7.50 %, and swapping its own untouched segmentation back in must change nothing. No strong prior for B.

**Setup:** T4 GPU · Colab secret `HF_TOKEN` · terms accepted for `pyannote/segmentation-3.0`, `pyannote/wespeaker-voxceleb-resnet34-LM`, `pyannote/speaker-diarization-community-1`.
Put `E009_upload.tgz` (242 MB) in **My Drive** (root). With `USE_DRIVE = True`, every checkpoint and result is saved to `My Drive/C3_E009` as it finishes, so a disconnect loses nothing. Re-running a cell skips finished work.

**Budget (estimate, not measured on a GPU):** Part A ≈ 30–60 min on a T4, Part B about the same again. A CPU dry run gave 6 training steps/epoch for `small` (same as E008) and 21 for `big`.
**If a cell errors:** paste the FULL error into the chat. Don't improvise around it.


In [ ]:
# Cell 1 — GPU check (should say T4 or better)
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
# Cell 2 — install, then Runtime > Restart session, continue from Cell 3
# Deliberately NOT pinned (logbook trap #7). The version actually used is recorded in Cell 3.
!pip install -q pyannote.audio pyannote.metrics
print('NOW: Runtime > Restart session, then continue from Cell 3.')


In [ ]:
# Cell 3 — config, unpack the bundle, build the two training protocols
import os, subprocess, json, time
from importlib.metadata import version
import torch

ROOT = '/content'
USE_DRIVE = True          # bundle read from, and every result saved to, Google Drive
SEEDS = [0, 1, 2]         # 3 training runs per arm; their spread is the noise floor
MAX_EPOCHS, PATIENCE = 40, 10    # identical to E008
RUN_PART_B = True         # community-1 arm; only runs after Part A is saved

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BUNDLE, WORK = '/content/drive/MyDrive/E009_upload.tgz', '/content/drive/MyDrive/C3_E009'
else:
    BUNDLE, WORK = f'{ROOT}/E009_upload.tgz', f'{ROOT}/e009'
os.makedirs(WORK, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if not os.path.exists(f'{ROOT}/e008_best.ckpt'):
    if not os.path.exists(BUNDLE):
        raise SystemExit(f'Bundle not found at {BUNDLE}. Put E009_upload.tgz there first.')
    subprocess.run(['tar', 'xzf', BUNDLE, '-C', ROOT], check=True)

TEST = ['IRD_Harshana_Silva', 'IRD_Recording_2_IIT']                 # FROZEN (E005/E008)
DEV = ['R0023', 'R0024', 'R0025', 'R0026']                            # early stopping only
TRAIN_SMALL = ['J26DS313_R0002', 'J26DS313_R0011', 'J26DS313_R0012',  # = E008's 21 clips
               'J26DS313_R0013', 'J26DS313_R0014', 'J26DS313_R0016',
               'R0003', 'R0015', 'R0017', 'R0018', 'R0019', 'R0020', 'R0021', 'R0022',
               'Segment-1', 'Segment-2', 'Segment-3', 'Segment-4', 'Segment-5',
               'Segment-6', 'Segment-7']
NEW = [f'R{n:04d}' for n in range(29, 53)]                            # R0029-R0052, the new batch
TRAIN_BIG = TRAIN_SMALL + NEW

# sanity: 51 distinct clips, each with audio + truth + UEM
allc = TEST + DEV + TRAIN_SMALL + NEW
assert len(allc) == len(set(allc)) == 51, 'a clip is in two splits, or one is missing'
missing = [f for c in allc for f in (f'{ROOT}/processed/{c}.wav', f'{ROOT}/rttm_truth/{c}.rttm',
                                     f'{ROOT}/rttm_truth/{c}.uem') if not os.path.exists(f)]
if missing:
    raise SystemExit(f'{len(missing)} files missing, e.g. {missing[:3]}')

def minutes(clips):
    tot = 0.0
    for c in clips:
        for line in open(f'{ROOT}/rttm_truth/{c}.uem', encoding='utf-8'):
            p = line.split()
            if len(p) >= 4:
                tot += float(p[3]) - float(p[2])
    return tot / 60
for name, clips, want in (('TRAIN small (=E008)', TRAIN_SMALL, 36.4), ('TRAIN big (E009)', TRAIN_BIG, 115.2),
                          ('DEV', DEV, 10.7), ('TEST (frozen)', TEST, 5.1)):
    m = minutes(clips)
    print(f'{name:<20} {len(clips):>3} clips {m:7.1f} min   (expected {want})')
    assert abs(m - want) < 0.1, f'{name}: minutes do not match the bundle that was prepared'

# The training loader needs strict 10-field RTTM. Three truth lines in the new batch still carry
# review notes (R0047, R0048, R0050: "check short(0.14s)" / "check inside-spk02"). They are passed
# through AS ANNOTATED (segment kept, note dropped) and listed here, never silently.
flagged = []
def clean_rttm(c):
    out = []
    for line in open(f'{ROOT}/rttm_truth/{c}.rttm', encoding='utf-8'):
        p = line.split()
        if len(p) >= 8 and p[0] == 'SPEAKER':
            if len(p) != 10:
                flagged.append(f'{c}: {" ".join(p[3:5])} {" ".join(p[7:])}')
            out.append(' '.join(p[:8] + ['<NA>', '<NA>']) + '\n')
    return ''.join(out)

os.makedirs(f'{ROOT}/db/lists', exist_ok=True)
for name, clips in (('small_train', TRAIN_SMALL), ('big_train', TRAIN_BIG), ('dev', DEV)):
    with open(f'{ROOT}/db/lists/{name}.lst', 'w') as fh:
        fh.write('\n'.join(clips) + '\n')
    with open(f'{ROOT}/db/lists/{name}.rttm', 'w') as fh:
        for c in clips:
            fh.write(clean_rttm(c))
    with open(f'{ROOT}/db/lists/{name}.uem', 'w') as fh:
        for c in clips:
            fh.write(open(f'{ROOT}/rttm_truth/{c}.uem', encoding='utf-8').read())
print('truth lines carrying unresolved review notes (kept as annotated):',
      *sorted(set(flagged)) or ['none'], sep='\n  ')

with open(f'{ROOT}/db/database.yml', 'w') as fh:
    fh.write(f'''Databases:
  C3: {ROOT}/processed/{{uri}}.wav
Protocols:
  C3:
    SpeakerDiarization:
      small:
        train:
          uri: {ROOT}/db/lists/small_train.lst
          annotation: {ROOT}/db/lists/small_train.rttm
          annotated: {ROOT}/db/lists/small_train.uem
        development:
          uri: {ROOT}/db/lists/dev.lst
          annotation: {ROOT}/db/lists/dev.rttm
          annotated: {ROOT}/db/lists/dev.uem
      big:
        train:
          uri: {ROOT}/db/lists/big_train.lst
          annotation: {ROOT}/db/lists/big_train.rttm
          annotated: {ROOT}/db/lists/big_train.uem
        development:
          uri: {ROOT}/db/lists/dev.lst
          annotation: {ROOT}/db/lists/dev.rttm
          annotated: {ROOT}/db/lists/dev.uem
''')

versions = {k: version(k) for k in ('pyannote.audio', 'pyannote.metrics', 'pyannote.core', 'torch')}
versions['gpu'] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
json.dump(versions, open(f'{WORK}/versions.json', 'w'), indent=1)
print('\n', versions)
if versions['pyannote.audio'] != '4.0.7':
    print('NOTE: E008 ran on pyannote.audio 4.0.7. Comparisons inside this run stay valid;'
          ' the reproduction check in Cell 7 shows whether old numbers drifted.')


In [ ]:
# Cell 4 — scoring (protocol v1, identical to E004/E008) + a resumable results store
import csv
from pyannote.core import Annotation, Segment, Timeline
from pyannote.metrics.diarization import DiarizationErrorRate
try:
    from google.colab import userdata
    HF = userdata.get('HF_TOKEN')          # trap #1: the name lives in code, the value in the secret store
except ImportError:
    HF = os.environ.get('HF_TOKEN')

def load_rttm(path, uri):
    ann = Annotation(uri=uri)
    for i, line in enumerate(open(path, encoding='utf-8')):
        p = line.split()
        if len(p) >= 8 and p[0] == 'SPEAKER' and float(p[4]) > 0:
            ann[Segment(float(p[3]), float(p[3]) + float(p[4])), f't{i}'] = p[7]
    return ann

def load_uem(path, uri):
    tl = Timeline(uri=uri)
    for line in open(path, encoding='utf-8'):
        p = line.split()
        if len(p) >= 4:
            tl.add(Segment(float(p[2]), float(p[3])))
    return tl

def score(hyps, clips, collar=0.0):
    pooled = dict.fromkeys(('confusion', 'missed detection', 'false alarm', 'total'), 0.0)
    per = {}
    for c in clips:
        ref = load_rttm(f'{ROOT}/rttm_truth/{c}.rttm', c)
        uem = load_uem(f'{ROOT}/rttm_truth/{c}.uem', c)
        comp = DiarizationErrorRate(collar=collar, skip_overlap=False)(ref, hyps[c], uem=uem, detailed=True)
        per[c] = sum(comp[k] for k in ('confusion', 'missed detection', 'false alarm')) / comp['total'] * 100
        for k in pooled:
            pooled[k] += comp[k]
    t = pooled['total']
    der = sum(pooled[k] for k in ('confusion', 'missed detection', 'false alarm')) / t * 100
    return der, pooled['missed detection'] / t * 100, pooled['false alarm'] / t * 100, pooled['confusion'] / t * 100, per

RES = f'{WORK}/e009_results.csv'
FIELDS = ['part', 'arm', 'seed', 'set', 'collar', 'DER', 'miss', 'fa', 'conf', 'per_clip']

def scored(part, arm, seed, setname):
    if not os.path.exists(RES):
        return False
    return any(r['part'] == part and r['arm'] == arm and r['seed'] == str(seed) and r['set'] == setname
               for r in csv.DictReader(open(RES)))

def get(part, arm, seed, setname, collar=0.0, field='DER'):
    for r in csv.DictReader(open(RES)):
        if (r['part'], r['arm'], r['seed'], r['set']) == (part, arm, str(seed), setname) and float(r['collar']) == collar:
            return float(r[field])
    return None

def run_and_score(part, arm, seed, pipe):
    for setname, clips in (('DEV', DEV), ('TEST', TEST)):
        if scored(part, arm, seed, setname):
            print(f'{part} {arm} s{seed} {setname}: already scored, skipped')
            continue
        outdir = f'{WORK}/rttm/{part}_{arm}_s{seed}'
        os.makedirs(outdir, exist_ok=True)
        hyps = {}
        for c in clips:
            out = pipe(f'{ROOT}/processed/{c}.wav')
            ann = out.speaker_diarization if hasattr(out, 'speaker_diarization') else out
            hyps[c] = ann
            with open(f'{outdir}/{c}.rttm', 'w') as fh:
                ann.write_rttm(fh)
        new_file = not os.path.exists(RES)
        with open(RES, 'a', newline='') as fh:
            w = csv.DictWriter(fh, fieldnames=FIELDS)
            if new_file:
                w.writeheader()
            for collar in (0.0, 0.25):
                der, miss, fa, conf, per = score(hyps, clips, collar)
                w.writerow(dict(part=part, arm=arm, seed=seed, set=setname, collar=collar,
                                DER=f'{der:.4f}', miss=f'{miss:.4f}', fa=f'{fa:.4f}', conf=f'{conf:.4f}',
                                per_clip=json.dumps({k: round(v, 4) for k, v in per.items()})))
                print(f'{part} {arm:>13} s{seed} {setname:<4} collar {collar}: DER {der:6.2f} '
                      f'(miss {miss:.2f} fa {fa:.2f} conf {conf:.2f})')
print('helpers ready; results go to', RES)


In [ ]:
# Cell 5 — the training function: E008's recipe, plus a fixed seed
import warnings
try:
    import lightning.pytorch as pl                     # what pyannote 4.x uses
    from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping
except ImportError:
    import pytorch_lightning as pl
    from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from pyannote.audio import Model
from pyannote.audio.tasks import SpeakerDiarization as SegTask
from pyannote.database import registry, FileFinder

os.environ['PYANNOTE_DATABASE_CONFIG'] = f'{ROOT}/db/database.yml'
registry.load_database(f'{ROOT}/db/database.yml')

def protocol(arm):
    return registry.get_protocol(f'C3.SpeakerDiarization.{arm}',
                                 preprocessors={'audio': FileFinder(registry=registry)})

def load_base(base):
    return Model.from_pretrained(base['checkpoint'], subfolder=base.get('subfolder'), token=HF)

TRAIN_LOG = f'{WORK}/train_log.csv'

def train(part, arm, seed, base):
    out = f'{WORK}/ckpt/{part}_{arm}_s{seed}'
    best = f'{out}/best.ckpt'
    if os.path.exists(best):
        print(f'{part} {arm} s{seed}: checkpoint exists, skipped')
        return best
    pl.seed_everything(seed, workers=True)
    model = load_base(base)                               # fresh pretrained weights every time
    spec = model.specifications
    # read the chunk/speaker limits FROM the model, so the task can never silently rebuild its
    # output layer (segmentation-3.0: 10 s, 3 per chunk, 2 per frame -- exactly E008's values)
    task = SegTask(protocol(arm), duration=float(spec.duration),
                   max_speakers_per_chunk=len(spec.classes),
                   max_speakers_per_frame=spec.powerset_max_classes,
                   batch_size=32, num_workers=2)
    model.task = task
    try:
        monitor, mode = task.val_monitor
    except Exception:
        monitor, mode = 'loss/val', 'min'
    ckpt = ModelCheckpoint(dirpath=out, filename='best', monitor=monitor, mode=mode, save_top_k=1)
    stop = EarlyStopping(monitor=monitor, mode=mode, patience=PATIENCE)
    trainer = pl.Trainer(accelerator='gpu' if torch.cuda.is_available() else 'cpu', devices=1,
                         max_epochs=MAX_EPOCHS, callbacks=[ckpt, stop], gradient_clip_val=0.5,
                         default_root_dir=f'{WORK}/logs/{part}_{arm}_s{seed}', enable_model_summary=False)
    t0 = time.time()
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        trainer.fit(model)
    if any('different task' in str(w.message) for w in caught):
        raise SystemExit('The output layer was re-initialised (task specs != model specs). '
                         'Paste this cell output into the chat.')
    if not os.path.exists(best):
        raise SystemExit(f'no checkpoint written to {best}. Paste this cell output into the chat.')
    row = dict(part=part, arm=arm, seed=seed, monitor=monitor,
               best_val=f'{float(ckpt.best_model_score):.4f}',
               best_epoch=torch.load(best, map_location='cpu', weights_only=False)['epoch'],
               last_epoch=trainer.current_epoch,
               steps_per_epoch=getattr(trainer, 'num_training_batches', ''),
               minutes=f'{(time.time() - t0) / 60:.1f}')
    new_file = not os.path.exists(TRAIN_LOG)
    with open(TRAIN_LOG, 'a', newline='') as fh:
        w = csv.DictWriter(fh, fieldnames=list(row))
        if new_file:
            w.writeheader()
        w.writerow(row)
    print('TRAINED', row)
    return best
print('train() ready')


In [ ]:
# Cell 6 — PART A training: segmentation-3.0 fine-tuned on small vs big, one run per seed
BASE_A = {'checkpoint': 'pyannote/segmentation-3.0'}
CKPT_A = {}
for seed in SEEDS:
    for arm in ('small', 'big'):
        CKPT_A[(arm, seed)] = train('A', arm, seed, BASE_A)
print(open(TRAIN_LOG).read())


In [ ]:
# Cell 7 — PART A scoring: every model in E008's pipeline (only the segmentation weights differ)
from pyannote.audio.pipelines import SpeakerDiarization as SDPipeline
EMB = 'pyannote/wespeaker-voxceleb-resnet34-LM'           # identical to E008

def build_A(seg):
    p = SDPipeline(segmentation=seg, embedding=EMB, token=HF)
    p.instantiate(p.default_parameters())                 # what E008 used on 4.0.7
    p.to(DEVICE)
    return p

arms = [('pretrained', '-', 'pyannote/segmentation-3.0'),
        ('e008_original', '-', f'{ROOT}/e008_best.ckpt')]    # the actual E008 model, re-scored here
arms += [(arm, seed, CKPT_A[(arm, seed)]) for seed in SEEDS for arm in ('small', 'big')]
for arm, seed, seg in arms:
    if scored('A', arm, seed, 'DEV') and scored('A', arm, seed, 'TEST'):
        print(f'A {arm} s{seed}: already scored, skipped')
        continue
    run_and_score('A', arm, seed, build_A(seg if seg.startswith('pyannote/') else Model.from_pretrained(seg)))
    torch.cuda.empty_cache()

print('\nREPRODUCTION CHECK vs the logbook (E008, 25 Aug, pyannote.audio 4.0.7), collar 0.0:')
for arm, setname, want in (('pretrained', 'TEST', 14.3473), ('e008_original', 'TEST', 16.7615),
                           ('pretrained', 'DEV', 22.4625), ('e008_original', 'DEV', 8.9897)):
    got = get('A', arm, '-', setname)
    tag = 'REPRODUCED' if abs(got - want) < 0.05 else f'DIFFERS by {got - want:+.2f} (drift; within-run comparisons stay valid)'
    print(f'  {arm:<14} {setname:<4} now {got:6.2f}  logbook {want:6.2f}  -> {tag}')


In [ ]:
# Cell 8 — PART A verdict, by the pre-registered rule
from math import comb

def verdict(part, refs):
    rows = [r for r in csv.DictReader(open(RES)) if r['part'] == part]
    pick = lambda arm, st, col=0.0: [r for r in rows if r['arm'] == arm and r['set'] == st and float(r['collar']) == col]
    lines = [f'===== PART {part} — TEST (frozen, {", ".join(TEST)}), collar 0.0 =====',
             f'{"arm":<14}{"seed":>5}{"DER":>8}{"miss":>7}{"fa":>7}{"conf":>7}   per clip']
    for arm, seed in refs:
        for r in pick(arm, 'TEST'):
            if r['seed'] == seed:
                lines.append(f'{arm:<14}{seed:>5}{float(r["DER"]):8.2f}{float(r["miss"]):7.2f}{float(r["fa"]):7.2f}{float(r["conf"]):7.2f}   {r["per_clip"]}')
    ders = {}
    for arm in ('small', 'big'):
        rs = sorted(pick(arm, 'TEST'), key=lambda r: r['seed'])
        for r in rs:
            lines.append(f'{arm:<14}{r["seed"]:>5}{float(r["DER"]):8.2f}{float(r["miss"]):7.2f}{float(r["fa"]):7.2f}{float(r["conf"]):7.2f}   {r["per_clip"]}')
        ders[arm] = [float(r['DER']) for r in rs]
        if rs:
            m = lambda f: sum(float(r[f]) for r in rs) / len(rs)
            lines.append(f'{arm + " MEAN":<14}{"":>5}{m("DER"):8.2f}{m("miss"):7.2f}{m("fa"):7.2f}{m("conf"):7.2f}'
                         f'   range {min(ders[arm]):.2f}-{max(ders[arm]):.2f}')
    s, b = ders.get('small', []), ders.get('big', [])
    if s and b:
        p = 1 / comb(len(s) + len(b), len(b))
        if max(b) < min(s):
            v = f'MORE DATA HELPED: every big run beat every small run (exact permutation p = {p:.3f}; training noise only, TEST n=2 clips)'
        elif min(b) > max(s):
            v = f'MORE DATA HURT: every big run was worse than every small run (p = {p:.3f})'
        else:
            v = 'NO DETECTABLE DIFFERENCE at this test size: the seed ranges overlap'
        lines += ['', 'VERDICT (pre-registered rule): ' + v,
                  f'mean change big - small: {sum(b)/len(b) - sum(s)/len(s):+.2f} pp DER']
        for arm, seed in refs:
            ref = get(part, arm, seed, 'TEST')
            if ref is not None:
                lines.append(f'big mean vs {arm}: {sum(b)/len(b) - ref:+.2f} pp  ({"better" if sum(b)/len(b) < ref else "worse"})')
    lines += ['', f'DEV (in-domain, speakers shared with training -> leakage-inflated, sanity only), collar 0.0:']
    for arm in [a for a, _ in refs] + ['small', 'big']:
        d = [float(r['DER']) for r in pick(arm, 'DEV')]
        if d:
            lines.append(f'  {arm:<14} mean {sum(d)/len(d):6.2f}  (n={len(d)})')
    text = '\n'.join(lines)
    open(f'{WORK}/e009_summary_{part}.txt', 'w').write(text + '\n')
    print(text)

verdict('A', refs=[('pretrained', '-'), ('e008_original', '-')])


In [ ]:
# Cell 9 — download Part A (results + RTTMs; checkpoints stay in WORK)
import zipfile, glob
def pack(name):
    zpath = f'{ROOT}/{name}.zip'
    with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
        for f in (['e009_results.csv', 'train_log.csv', 'versions.json'] + [os.path.basename(x) for x in glob.glob(f'{WORK}/e009_summary_*.txt')]):
            if os.path.exists(f'{WORK}/{f}'):
                z.write(f'{WORK}/{f}', f)
        for f in glob.glob(f'{WORK}/rttm/*/*.rttm'):
            z.write(f, os.path.relpath(f, WORK))
    return zpath
zp = pack('e009_partA')
try:
    from google.colab import files
    files.download(zp)
except ImportError:
    print('saved', zp)
# Send this zip to the chat. The logbook entry gets written from it.


## Part B (optional) — the same question inside community-1

Part A answers "does more data help?" in E008's pipeline, which the logbook showed is **2× worse on real audio** than community-1 (Finding 4, 25 Aug).
Part B asks the question that matters for the deliverable: **can our data improve community-1 itself?**

The method: fine-tune community-1's own segmentation model on `small` vs `big` (same seeds, same recipe), then swap it into an otherwise untouched community-1.
Two controls must pass first, or the cell stops:
- **B0:** community-1 as shipped must reproduce E004 on TEST (27.99 / 4.87, pooled 7.50 %).
- **B1:** swapping community-1's own, untrained segmentation back in must give exactly B0. This proves the swap mechanism changes nothing by itself.

Skip it (set `RUN_PART_B = False` in Cell 3) if Colab time is short. Part A is already saved.


In [ ]:
# Cell 10 — PART B: controls, then small vs big inside community-1
if not RUN_PART_B:
    raise SystemExit('Part B skipped (RUN_PART_B = False).')
from pyannote.audio import Pipeline, Inference
C1 = 'pyannote/speaker-diarization-community-1'
BASE_B = {'checkpoint': C1, 'subfolder': 'segmentation'}

def fresh_c1():
    p = Pipeline.from_pretrained(C1, token=HF)
    p.to(DEVICE)
    return p

def swap_segmentation(p, model):
    # rebuild the segmentation Inference exactly as SpeakerDiarization.__init__ does, around a new model
    old = p._segmentation
    assert model.specifications.powerset == old.model.specifications.powerset, 'segmentation types differ'
    d = model.specifications.duration
    p._segmentation = Inference(model, duration=d, step=p.segmentation_step * d,
                                skip_aggregation=True, batch_size=old.batch_size)
    p.to(DEVICE)
    return p

# B0 — community-1 exactly as shipped
run_and_score('B', 'c1_zeroshot', '-', fresh_c1())

# B1 — community-1 with its OWN segmentation swapped back in
own = load_base(BASE_B)
c1 = fresh_c1()
sd_own, sd_c1 = own.state_dict(), c1._segmentation.model.state_dict()
same_w = sd_own.keys() == sd_c1.keys() and all(torch.equal(sd_own[k].cpu(), sd_c1[k].cpu()) for k in sd_own)
run_and_score('B', 'c1_selfswap', '-', swap_segmentation(c1, own))
b0, b1 = get('B', 'c1_zeroshot', '-', 'TEST'), get('B', 'c1_selfswap', '-', 'TEST')
print(f'\nloaded segmentation weights == shipped: {same_w}')
print(f'B0 community-1 TEST {b0:.4f}   (E004 logbook: 7.4980)')
print(f'B1 self-swap    TEST {b1:.4f}   (must equal B0)')
if not same_w or abs(b0 - b1) > 1e-6 or abs(get('B', 'c1_zeroshot', '-', 'DEV') - get('B', 'c1_selfswap', '-', 'DEV')) > 1e-6:
    raise SystemExit('SWAP CONTROL FAILED: Part B would not be trustworthy. Paste this output into the chat.')
print('swap control PASSED')
if abs(b0 - 7.498) > 0.05:
    print('NOTE: community-1 TEST differs from E004. That is drift; comparisons inside this run stay valid.')

# fine-tune community-1's segmentation on small vs big, then score inside community-1
CKPT_B = {}
for seed in SEEDS:
    for arm in ('small', 'big'):
        CKPT_B[(arm, seed)] = train('B', arm, seed, BASE_B)
for seed in SEEDS:
    for arm in ('small', 'big'):
        if scored('B', arm, seed, 'DEV') and scored('B', arm, seed, 'TEST'):
            print(f'B {arm} s{seed}: already scored, skipped')
            continue
        run_and_score('B', arm, seed, swap_segmentation(fresh_c1(), Model.from_pretrained(CKPT_B[(arm, seed)])))
        torch.cuda.empty_cache()


In [ ]:
# Cell 11 — PART B verdict (same rule), then download everything
verdict('B', refs=[('c1_zeroshot', '-')])
zp = pack('e009_all')
try:
    from google.colab import files
    files.download(zp)
except ImportError:
    print('saved', zp)
# Send e009_all.zip to the chat.


## What to send back
`e009_partA.zip` after Cell 9 (and `e009_all.zip` after Cell 11, if Part B ran). The logbook entry is written from them.
The trained checkpoints stay in `My Drive/C3_E009/ckpt/`. Keep them.
